# Baseline linear multialvo direto — configuração TU

Este notebook implementa a primeira modelagem da segunda etapa do Projeto Integrador IV.

## Objetivo

Comparar dois referenciais para prever diretamente **temperatura do ar** e **umidade relativa** de `t+1` a `t+24`:

1. **Baseline sazonal de 24 horas**: repete, para cada horizonte, o valor observado 24 horas antes.
2. **Regressão Ridge multialvo direta**: recebe o histórico TU e os atributos temporais futuros e produz as 48 saídas em uma única operação.

## Protocolo

- A configuração de atributos fica congelada como **TU** nesta etapa.
- O hiperparâmetro `alpha` é escolhido somente com a validação de 2024.
- O teste de 2025 permanece intocado até o encerramento da seleção.
- 2026 parcial é apenas avaliação adicional.
- Não são utilizados valores meteorológicos futuros reais como entrada.
- As métricas são calculadas nas unidades originais: °C e pontos percentuais de umidade.

## 1. Importações e configuração

O diretório deve apontar para a pasta `TU` produzida pelo notebook de preparação.

In [1]:
from pathlib import Path
import json
import platform
import sys
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

DIRETORIO_TU = Path("dados_processados_multivariados/TU")
DIRETORIO_SAIDA = Path("resultados_modelagem/linear_TU")
DIRETORIO_SAIDA.mkdir(parents=True, exist_ok=True)

SPLITS = ["treino", "validacao", "teste", "avaliacao_2026"]
HORIZONTES_DESTAQUE = [1, 3, 6, 12, 24]
ALPHAS = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0]
SEMENTE = 42

ARQUIVOS_OBRIGATORIOS = [
    DIRETORIO_TU / "treino.npz",
    DIRETORIO_TU / "validacao.npz",
    DIRETORIO_TU / "teste.npz",
    DIRETORIO_TU / "avaliacao_2026.npz",
    DIRETORIO_TU / "scaler_x.joblib",
    DIRETORIO_TU / "scaler_y.joblib",
    DIRETORIO_TU / "metadados.json",
]

faltantes = [str(p) for p in ARQUIVOS_OBRIGATORIOS if not p.exists()]
if faltantes:
    raise FileNotFoundError(
        "Arquivos obrigatórios não encontrados:" + chr(10) + "- " + (chr(10) + "- ").join(faltantes)
    )

print("Entrada:", DIRETORIO_TU.resolve())
print("Saída:", DIRETORIO_SAIDA.resolve())

Entrada: C:\programas\univesp\pic-iv\dados_processados_multivariados\TU
Saída: C:\programas\univesp\pic-iv\resultados_modelagem\linear_TU


## 2. Carregamento seguro dos artefatos

As datas são mantidas para auditoria. As matrizes são carregadas em memória porque a regressão linear requer acesso ao conjunto completo de treinamento.

In [2]:
def carregar_split(nome_split: str) -> dict:
    caminho = DIRETORIO_TU / f"{nome_split}.npz"
    with np.load(caminho) as arq:
        return {chave: arq[chave] for chave in arq.files}

with (DIRETORIO_TU / "metadados.json").open("r", encoding="utf-8") as f:
    metadados_preparacao = json.load(f)

scaler_x = joblib.load(DIRETORIO_TU / "scaler_x.joblib")
scaler_y = joblib.load(DIRETORIO_TU / "scaler_y.joblib")

dados = {nome: carregar_split(nome) for nome in SPLITS}

assert metadados_preparacao["configuracao"] == "TU"
assert metadados_preparacao["horizonte"] == 24
assert metadados_preparacao["tau"] == 24
assert metadados_preparacao["usa_covariaveis_meteorologicas_futuras"] is False
assert metadados_preparacao["colunas_alvo"] == [
    "temperatura_ar_horaria_c",
    "umidade_horaria_pct",
]

resumo_formas = []
for nome, split in dados.items():
    resumo_formas.append({
        "split": nome,
        "X_historico": str(split["X_historico"].shape),
        "X_futuro_temporal": str(split["X_futuro_temporal"].shape),
        "y": str(split["y"].shape),
        "inicio": str(split["inicio_previsao"].min()),
        "fim": str(split["fim_previsao"].max()),
    })

pd.DataFrame(resumo_formas)

,split,X_historico,X_futuro_temporal,y,inicio,fim
0,treino,"(84744, 24, 6)","(84744, 24, 4)","(84744, 24, 2)",2011-01-02T00:00:00.000000000,2023-12-31T23:00:00.000000000
1,validacao,"(7837, 24, 6)","(7837, 24, 4)","(7837, 24, 2)",2024-01-02T00:00:00.000000000,2024-12-31T23:00:00.000000000
2,teste,"(7970, 24, 6)","(7970, 24, 4)","(7970, 24, 2)",2025-01-02T00:00:00.000000000,2025-12-31T23:00:00.000000000
3,avaliacao_2026,"(4551, 24, 6)","(4551, 24, 4)","(4551, 24, 2)",2026-01-02T00:00:00.000000000,2026-08-14T05:00:00.000000000


## 3. Construção da matriz tabular

A regressão Ridge recebe uma matriz bidimensional. Portanto:

- `X_historico` é achatado preservando a ordem temporal e dos atributos;
- `X_futuro_temporal` também é achatado;
- ambos são concatenados;
- o alvo `(24, 2)` é convertido em 48 saídas, na ordem horizonte × alvo.

Nenhum valor meteorológico futuro é acrescentado.

In [3]:
def preparar_X(split: dict) -> np.ndarray:
    X_hist = split["X_historico"].reshape(len(split["X_historico"]), -1)
    X_fut = split["X_futuro_temporal"].reshape(len(split["X_futuro_temporal"]), -1)
    X = np.concatenate([X_hist, X_fut], axis=1)
    if not np.isfinite(X).all():
        raise ValueError("A matriz de entrada contém NaN ou infinito.")
    return X.astype(np.float32, copy=False)


def preparar_y(split: dict) -> np.ndarray:
    y = split["y"].reshape(len(split["y"]), -1)
    if not np.isfinite(y).all():
        raise ValueError("A matriz alvo contém NaN ou infinito.")
    return y.astype(np.float32, copy=False)

X = {nome: preparar_X(split) for nome, split in dados.items()}
y = {nome: preparar_y(split) for nome, split in dados.items()}

n_atributos_esperado = (
    dados["treino"]["X_historico"].shape[1] * dados["treino"]["X_historico"].shape[2]
    + dados["treino"]["X_futuro_temporal"].shape[1] * dados["treino"]["X_futuro_temporal"].shape[2]
)
assert X["treino"].shape[1] == n_atributos_esperado
assert y["treino"].shape[1] == 24 * 2

print("Atributos tabulares:", X["treino"].shape[1])
print("Saídas diretas:", y["treino"].shape[1])

Atributos tabulares: 240
Saídas diretas: 48


## 4. Funções de inversão e métricas

As métricas principais são:

- MAE e RMSE globais por variável;
- MAE e RMSE por horizonte;
- viés médio por variável e horizonte;
- resultados em `t+1`, `t+3`, `t+6`, `t+12` e `t+24`;
- média acumulada até os mesmos horizontes.

Temperatura e umidade são reportadas separadamente porque possuem unidades e escalas diferentes.

In [4]:
NOMES_ALVOS = ["temperatura_c", "umidade_pct"]


def remodelar_y(y_2d: np.ndarray) -> np.ndarray:
    return y_2d.reshape(-1, 24, 2)


def inverter_y(y_normalizado: np.ndarray) -> np.ndarray:
    forma = y_normalizado.shape
    y_2d = y_normalizado.reshape(-1, 2)
    y_original = scaler_y.inverse_transform(y_2d)
    return y_original.reshape(forma)


def metricas_detalhadas(y_real: np.ndarray, y_pred: np.ndarray, modelo: str, split: str) -> pd.DataFrame:
    linhas = []
    for h in range(24):
        for j, alvo in enumerate(NOMES_ALVOS):
            real = y_real[:, h, j]
            pred = y_pred[:, h, j]
            linhas.append({
                "modelo": modelo,
                "split": split,
                "horizonte_horas": h + 1,
                "alvo": alvo,
                "mae": mean_absolute_error(real, pred),
                "rmse": mean_squared_error(real, pred) ** 0.5,
                "vies": float(np.mean(pred - real)),
                "n": len(real),
            })
    return pd.DataFrame(linhas)


def resumo_global(y_real: np.ndarray, y_pred: np.ndarray, modelo: str, split: str) -> pd.DataFrame:
    linhas = []
    for j, alvo in enumerate(NOMES_ALVOS):
        real = y_real[:, :, j].ravel()
        pred = y_pred[:, :, j].ravel()
        linhas.append({
            "modelo": modelo,
            "split": split,
            "alvo": alvo,
            "mae_global_1a24": mean_absolute_error(real, pred),
            "rmse_global_1a24": mean_squared_error(real, pred) ** 0.5,
            "vies_global_1a24": float(np.mean(pred - real)),
            "n_previsoes": len(real),
        })
    return pd.DataFrame(linhas)


def resumo_horizontes(metricas: pd.DataFrame) -> pd.DataFrame:
    exatos = metricas[metricas["horizonte_horas"].isin(HORIZONTES_DESTAQUE)].copy()
    exatos["tipo"] = "exato"

    acumulados = []
    for (modelo, split, alvo), grupo in metricas.groupby(["modelo", "split", "alvo"]):
        for h in HORIZONTES_DESTAQUE:
            trecho = grupo[grupo["horizonte_horas"] <= h]
            acumulados.append({
                "modelo": modelo,
                "split": split,
                "alvo": alvo,
                "horizonte_horas": h,
                "mae": trecho["mae"].mean(),
                "rmse": trecho["rmse"].mean(),
                "vies": trecho["vies"].mean(),
                "n": trecho["n"].sum(),
                "tipo": "media_ate_h",
            })
    return pd.concat([exatos, pd.DataFrame(acumulados)], ignore_index=True)

## 5. Baseline sazonal de 24 horas

Como a janela histórica contém exatamente 24 horas, o baseline prevê para `t+h` o valor observado na mesma posição da janela anterior. Os dois primeiros canais históricos da configuração TU correspondem a temperatura e umidade normalizadas.

In [5]:
def prever_sazonal_24h(split: dict) -> np.ndarray:
    # Os canais 0 e 1 são temperatura e umidade normalizadas.
    return split["X_historico"][:, :, :2].copy()

metricas_sazonal_validacao = metricas_detalhadas(
    inverter_y(remodelar_y(y["validacao"])),
    inverter_y(prever_sazonal_24h(dados["validacao"])),
    modelo="sazonal_24h",
    split="validacao",
)

resumo_global_sazonal_validacao = resumo_global(
    inverter_y(remodelar_y(y["validacao"])),
    inverter_y(prever_sazonal_24h(dados["validacao"])),
    modelo="sazonal_24h",
    split="validacao",
)

resumo_global_sazonal_validacao

,modelo,split,alvo,mae_global_1a24,rmse_global_1a24,vies_global_1a24,n_previsoes
0,sazonal_24h,validacao,temperatura_c,2.085083,2.958672,-0.011233,188088
1,sazonal_24h,validacao,umidade_pct,10.837693,15.206184,-0.238505,188088


## 6. Seleção de `alpha` exclusivamente em 2024

`Ridge(alpha=0)` corresponde à regressão linear sem penalização. Os demais valores avaliam regularização L2. O critério principal é a média dos MAEs normalizados relativos à variabilidade de cada alvo no treino, evitando que a escala da umidade domine a temperatura.

O teste de 2025 não é consultado nesta seleção.

In [6]:
def criterio_validacao(y_real_original: np.ndarray, y_pred_original: np.ndarray) -> tuple[float, float, float]:
    desvios = scaler_y.scale_
    maes = []
    for j in range(2):
        mae = mean_absolute_error(y_real_original[:, :, j].ravel(), y_pred_original[:, :, j].ravel())
        maes.append(mae / desvios[j])
    return float(np.mean(maes)), float(maes[0]), float(maes[1])

resultados_alpha = []
modelos_candidatos = {}
y_val_original = inverter_y(remodelar_y(y["validacao"]))

for alpha in ALPHAS:
    inicio = time.perf_counter()
    modelo = Ridge(alpha=alpha, fit_intercept=True, solver="lsqr", tol=1e-4)
    modelo.fit(X["treino"], y["treino"])
    pred_val_norm = remodelar_y(modelo.predict(X["validacao"]))
    pred_val_original = inverter_y(pred_val_norm)
    criterio, criterio_temp, criterio_umid = criterio_validacao(y_val_original, pred_val_original)

    resultados_alpha.append({
        "alpha": alpha,
        "criterio_mae_padronizado_medio": criterio,
        "mae_padronizado_temperatura": criterio_temp,
        "mae_padronizado_umidade": criterio_umid,
        "tempo_treino_segundos": time.perf_counter() - inicio,
    })
    modelos_candidatos[alpha] = modelo

resultados_alpha = pd.DataFrame(resultados_alpha).sort_values(
    ["criterio_mae_padronizado_medio", "alpha"],
    ascending=[True, True],
).reset_index(drop=True)

MELHOR_ALPHA = float(resultados_alpha.loc[0, "alpha"])
modelo_linear = modelos_candidatos[MELHOR_ALPHA]

print("Alpha selecionado somente em 2024:", MELHOR_ALPHA)
resultados_alpha

Alpha selecionado somente em 2024: 1.0


,alpha,criterio_mae_padronizado_medio,mae_padronizado_temperatura,mae_padronizado_umidade,tempo_treino_segundos
0,1.00,0.378751,0.341426,0.416077,33.262487
1,0.10,0.378760,0.341452,0.416068,35.453602
2,0.01,0.378761,0.341455,0.416068,31.227373
3,0.00,0.378761,0.341455,0.416068,31.365044
4,10.00,0.378908,0.341735,0.416082,29.822589
5,100.00,0.379321,0.342453,0.416189,29.104446


## 7. Avaliação fechada em validação, teste e 2026 parcial

Após congelar `alpha`, o mesmo modelo treinado apenas em 2011–2023, excluindo 2022, é aplicado aos três subconjuntos. Não há retreinamento com validação ou teste nesta etapa, preservando a comparabilidade.

In [7]:
metricas_todas = []
resumos_globais = []
previsoes_salvas = {}

for nome_split in ["validacao", "teste", "avaliacao_2026"]:
    y_real_orig = inverter_y(remodelar_y(y[nome_split]))

    pred_linear_norm = remodelar_y(modelo_linear.predict(X[nome_split]))
    pred_linear_orig = inverter_y(pred_linear_norm)

    pred_sazonal_norm = prever_sazonal_24h(dados[nome_split])
    pred_sazonal_orig = inverter_y(pred_sazonal_norm)

    metricas_todas.extend([
        metricas_detalhadas(y_real_orig, pred_sazonal_orig, "sazonal_24h", nome_split),
        metricas_detalhadas(y_real_orig, pred_linear_orig, "ridge_direto_TU", nome_split),
    ])
    resumos_globais.extend([
        resumo_global(y_real_orig, pred_sazonal_orig, "sazonal_24h", nome_split),
        resumo_global(y_real_orig, pred_linear_orig, "ridge_direto_TU", nome_split),
    ])

    previsoes_salvas[nome_split] = {
        "y_real": y_real_orig.astype(np.float32),
        "y_pred_linear": pred_linear_orig.astype(np.float32),
        "y_pred_sazonal": pred_sazonal_orig.astype(np.float32),
        "inicio_previsao": dados[nome_split]["inicio_previsao"],
        "fim_previsao": dados[nome_split]["fim_previsao"],
    }

metricas_todas = pd.concat(metricas_todas, ignore_index=True)
resumos_globais = pd.concat(resumos_globais, ignore_index=True)
resumo_selecionado = resumo_horizontes(metricas_todas)

resumos_globais.sort_values(["split", "alvo", "modelo"])

,modelo,split,alvo,mae_global_1a24,rmse_global_1a24,vies_global_1a24,n_previsoes
10,ridge_direto_TU,avaliacao_2026,temperatura_c,1.514856,2.131869,0.036787,109224
8,sazonal_24h,avaliacao_2026,temperatura_c,1.920754,2.704827,0.054422,109224
11,ridge_direto_TU,avaliacao_2026,umidade_pct,7.307822,10.253129,-1.140046,109224
9,sazonal_24h,avaliacao_2026,umidade_pct,9.586437,13.479410,0.085174,109224
6,ridge_direto_TU,teste,temperatura_c,1.700491,2.401878,0.038398,191280
4,sazonal_24h,teste,temperatura_c,2.210932,3.100861,0.014382,191280
7,ridge_direto_TU,teste,umidade_pct,8.251848,11.632867,-0.536772,191280
5,sazonal_24h,teste,umidade_pct,11.175094,15.777979,-0.345159,191280
2,ridge_direto_TU,validacao,temperatura_c,1.696510,2.324284,-0.227446,188088
0,sazonal_24h,validacao,temperatura_c,2.085083,2.958672,-0.011233,188088


## 8. Comparação percentual com o baseline sazonal

Valores positivos representam redução de erro do modelo Ridge em relação ao baseline sazonal.

In [8]:
def calcular_ganho_percentual(resumos: pd.DataFrame) -> pd.DataFrame:
    saz = resumos[resumos["modelo"] == "sazonal_24h"].set_index(["split", "alvo"])
    lin = resumos[resumos["modelo"] == "ridge_direto_TU"].set_index(["split", "alvo"])
    linhas = []
    for indice in lin.index:
        split, alvo = indice
        linhas.append({
            "split": split,
            "alvo": alvo,
            "ganho_mae_pct": 100 * (saz.loc[indice, "mae_global_1a24"] - lin.loc[indice, "mae_global_1a24"]) / saz.loc[indice, "mae_global_1a24"],
            "ganho_rmse_pct": 100 * (saz.loc[indice, "rmse_global_1a24"] - lin.loc[indice, "rmse_global_1a24"]) / saz.loc[indice, "rmse_global_1a24"],
        })
    return pd.DataFrame(linhas)

ganhos = calcular_ganho_percentual(resumos_globais)
ganhos

,split,alvo,ganho_mae_pct,ganho_rmse_pct
0,validacao,temperatura_c,18.635885,21.441650
1,validacao,umidade_pct,23.794505,25.100121
2,teste,temperatura_c,23.087106,22.541561
3,teste,umidade_pct,26.158577,26.271503
4,avaliacao_2026,temperatura_c,21.132211,21.182789
5,avaliacao_2026,umidade_pct,23.769159,23.934884


## 9. Salvamento do modelo, métricas e previsões

Os artefatos registram o protocolo, o `alpha` escolhido e a origem dos dados processados. As previsões são salvas separadamente por split para permitir auditoria e gráficos posteriores sem repetir o treinamento.

In [9]:
joblib.dump(modelo_linear, DIRETORIO_SAIDA / "modelo_ridge_direto_TU.joblib")
resultados_alpha.to_csv(DIRETORIO_SAIDA / "selecao_alpha_validacao_2024.csv", index=False, encoding="utf-8-sig")
metricas_todas.to_csv(DIRETORIO_SAIDA / "metricas_por_horizonte.csv", index=False, encoding="utf-8-sig")
resumos_globais.to_csv(DIRETORIO_SAIDA / "metricas_globais.csv", index=False, encoding="utf-8-sig")
resumo_selecionado.to_csv(DIRETORIO_SAIDA / "metricas_horizontes_destaque.csv", index=False, encoding="utf-8-sig")
ganhos.to_csv(DIRETORIO_SAIDA / "ganhos_vs_sazonal.csv", index=False, encoding="utf-8-sig")

for nome_split, conteudo in previsoes_salvas.items():
    np.savez_compressed(DIRETORIO_SAIDA / f"previsoes_{nome_split}.npz", **conteudo)

metadados_modelo = {
    "modelo": "Ridge multialvo direto",
    "configuracao": "TU",
    "alpha_selecionado": MELHOR_ALPHA,
    "criterio_selecao": "menor média do MAE por alvo dividida pelo desvio do treino, calculada somente em 2024",
    "solver": "lsqr",
    "fit_intercept": True,
    "tau": 24,
    "horizonte": 24,
    "n_entradas_tabulares": int(X["treino"].shape[1]),
    "n_saidas": int(y["treino"].shape[1]),
    "alvos": ["temperatura_ar_horaria_c", "umidade_horaria_pct"],
    "ordem_saida": "horizonte primeiro, alvo depois: [t+1 temperatura, t+1 umidade, ..., t+24 temperatura, t+24 umidade]",
    "usa_covariaveis_meteorologicas_futuras": False,
    "usa_covariaveis_temporais_futuras": True,
    "treinado_com": "treino preparado: 2011-2023, excluindo 2022",
    "validacao": 2024,
    "teste": 2025,
    "avaliacao_adicional": "2026 parcial",
    "semente": SEMENTE,
    "metadados_preparacao": metadados_preparacao,
    "versoes": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "plataforma": platform.platform(),
    },
}

with (DIRETORIO_SAIDA / "metadados_modelo.json").open("w", encoding="utf-8") as f:
    json.dump(metadados_modelo, f, ensure_ascii=False, indent=2)

print("Artefatos salvos em:", DIRETORIO_SAIDA.resolve())

Artefatos salvos em: C:\programas\univesp\pic-iv\resultados_modelagem\linear_TU


## 10. Verificação final e resumo para revisão

A etapa é considerada concluída quando todos os artefatos existem, as métricas são finitas e nenhuma previsão utiliza valores meteorológicos futuros nas entradas.

In [10]:
arquivos_saida = [
    "modelo_ridge_direto_TU.joblib",
    "selecao_alpha_validacao_2024.csv",
    "metricas_por_horizonte.csv",
    "metricas_globais.csv",
    "metricas_horizontes_destaque.csv",
    "ganhos_vs_sazonal.csv",
    "metadados_modelo.json",
    "previsoes_validacao.npz",
    "previsoes_teste.npz",
    "previsoes_avaliacao_2026.npz",
]

assert all((DIRETORIO_SAIDA / nome).exists() for nome in arquivos_saida)
assert np.isfinite(metricas_todas[["mae", "rmse", "vies"]].to_numpy()).all()
assert np.isfinite(resumos_globais[["mae_global_1a24", "rmse_global_1a24", "vies_global_1a24"]].to_numpy()).all()
assert metadados_modelo["usa_covariaveis_meteorologicas_futuras"] is False

print("VERIFICAÇÃO FINAL: OK")
print("Alpha selecionado:", MELHOR_ALPHA)
print()
print("Métricas globais:")
display(resumos_globais.sort_values(["split", "alvo", "modelo"]).reset_index(drop=True))
print()
print("Ganhos percentuais do Ridge sobre o sazonal:")
display(ganhos.sort_values(["split", "alvo"]).reset_index(drop=True))
print()
print("Horizontes de destaque:")
display(
    resumo_selecionado[
        (resumo_selecionado["modelo"] == "ridge_direto_TU")
        & (resumo_selecionado["tipo"] == "exato")
    ].sort_values(["split", "alvo", "horizonte_horas"]).reset_index(drop=True)
)

VERIFICAÇÃO FINAL: OK
Alpha selecionado: 1.0

Métricas globais:


,modelo,split,alvo,mae_global_1a24,rmse_global_1a24,vies_global_1a24,n_previsoes
0,ridge_direto_TU,avaliacao_2026,temperatura_c,1.514856,2.131869,0.036787,109224
1,sazonal_24h,avaliacao_2026,temperatura_c,1.920754,2.704827,0.054422,109224
2,ridge_direto_TU,avaliacao_2026,umidade_pct,7.307822,10.253129,-1.140046,109224
3,sazonal_24h,avaliacao_2026,umidade_pct,9.586437,13.479410,0.085174,109224
4,ridge_direto_TU,teste,temperatura_c,1.700491,2.401878,0.038398,191280
5,sazonal_24h,teste,temperatura_c,2.210932,3.100861,0.014382,191280
6,ridge_direto_TU,teste,umidade_pct,8.251848,11.632867,-0.536772,191280
7,sazonal_24h,teste,umidade_pct,11.175094,15.777979,-0.345159,191280
8,ridge_direto_TU,validacao,temperatura_c,1.696510,2.324284,-0.227446,188088
9,sazonal_24h,validacao,temperatura_c,2.085083,2.958672,-0.011233,188088



Ganhos percentuais do Ridge sobre o sazonal:


,split,alvo,ganho_mae_pct,ganho_rmse_pct
0,avaliacao_2026,temperatura_c,21.132211,21.182789
1,avaliacao_2026,umidade_pct,23.769159,23.934884
2,teste,temperatura_c,23.087106,22.541561
3,teste,umidade_pct,26.158577,26.271503
4,validacao,temperatura_c,18.635885,21.441650
5,validacao,umidade_pct,23.794505,25.100121



Horizontes de destaque:


,modelo,split,horizonte_horas,alvo,mae,rmse,vies,n,tipo
0,ridge_direto_TU,avaliacao_2026,1,temperatura_c,0.545047,0.853381,0.007019,4551,exato
1,ridge_direto_TU,avaliacao_2026,3,temperatura_c,1.023804,1.480161,0.022433,4551,exato
2,ridge_direto_TU,avaliacao_2026,6,temperatura_c,1.370886,1.938430,0.033341,4551,exato
3,ridge_direto_TU,avaliacao_2026,12,temperatura_c,1.620711,2.216974,0.046787,4551,exato
4,ridge_direto_TU,avaliacao_2026,24,temperatura_c,1.786417,2.447528,0.035762,4551,exato
5,ridge_direto_TU,avaliacao_2026,1,umidade_pct,2.714446,4.237756,-0.136624,4551,exato
6,ridge_direto_TU,avaliacao_2026,3,umidade_pct,5.044756,7.170619,-0.406430,4551,exato
7,ridge_direto_TU,avaliacao_2026,6,umidade_pct,6.626571,9.162533,-0.746848,4551,exato
8,ridge_direto_TU,avaliacao_2026,12,umidade_pct,7.776823,10.650996,-1.250950,4551,exato
9,ridge_direto_TU,avaliacao_2026,24,umidade_pct,8.766022,11.981915,-1.698789,4551,exato


## Encerramento desta etapa

Antes de avançar para TUO, TUOV, gradient boosting ou GRU, revise:

- o `alpha` escolhido em 2024;
- o desempenho separado de temperatura e umidade;
- a evolução do erro entre `t+1` e `t+24`;
- o ganho ou perda em relação ao baseline sazonal;
- o comportamento em 2025 e 2026 parcial.

A configuração TU e o modelo Ridge constituem o **baseline multialvo direto** da segunda etapa.